In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 28. 데이터 병렬 (Data Parallel)

> **학습 목표**
> - 데이터 병렬화의 통신 패턴(All-Reduce)을 이해한다
> - 그래디언트 동기화 수학을 설명한다
> - DDP (DistributedDataParallel)를 사용해 본다

## 28.1 단일 GPU의 한계

LLM 학습은 메모리·연산량이 엄청남:
- 7B 모델 FP16 + AdamW = ~70GB
- 큰 배치 = 더 큰 메모리
- 속도: 1 GPU로는 수개월

해결: 여러 GPU로 분산.

## 28.2 데이터 병렬의 기본 아이디어

1. 같은 모델 복사를 $N$개 GPU에 배치
2. 데이터를 $N$등분하여 각 GPU에 할당
3. 각 GPU가 자기 데이터로 그래디언트 계산
4. 그래디언트를 모아서 평균 → 모든 GPU에 동기화
5. 각 GPU가 같은 업데이트 수행 → 모델 일관성 유지

수학:
- 로컬 그래디언트: $\mathbf{g}_i = \nabla \mathcal{L}_i(\theta)$
- 동기화: $\mathbf{g} = \frac{1}{N} \sum_{i=1}^{N} \mathbf{g}_i$
- 업데이트: $\theta \leftarrow \theta - \eta \mathbf{g}$

효과적 배치 크기: $B_{\text{eff}} = N \cdot B_{\text{local}}$


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# 데이터 병렬 시뮬레이션 (단일 GPU에서 gradient accumulation으로 흉내)
class DataParallelSimulator:
    """단일 GPU에서 N-GPU DDP를 흉내내는 시뮬레이터."""
    def __init__(self, model, n_gpus=4):
        self.model = model
        self.n_gpus = n_gpus
        # N개의 "가짜 GPU" = 같은 Model의 복사 (gradient accumulation으로 시뮬레이션)
        # 실제로는 한 Model에 gradient를 n_gpus번 누적
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    def step(self, x_batch, y_batch):
        """x_batch: (N*B, ...), N개 GPU 각각 B개 Sample."""
        # 1. Batch를 N등분
        chunks = torch.chunk(x_batch, self.n_gpus)
        y_chunks = torch.chunk(y_batch, self.n_gpus)

        # 2. 각 "GPU"에서 Gradient Computation (gradient accumulation)
        self.optimizer.zero_grad()
        total_loss = 0
        for i in range(self.n_gpus):
            out = self.model(chunks[i])
            loss = F.mse_loss(out, y_chunks[i]) / self.n_gpus  # Mean
            loss.backward()  # Gradient 누적
            total_loss += loss.item() * self.n_gpus

        # 3. optimizer.step()은 누적된 Gradient로 (자동으로 Mean됨)
        self.optimizer.step()
        return total_loss / self.n_gpus

# 작은 Model로 시뮬레이션
torch.manual_seed(0)
model = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Linear(64, 20)
)

dp = DataParallelSimulator(model, n_gpus=4)

# Comparison: 단일 Batch vs DDP 시뮬레이션
x = torch.randn(64, 20)
y = torch.randn(64, 20)

loss = dp.step(x, y)
print(f"DDP 시뮬레이션 1 step loss: {loss:.4f}")
print(f"n_gpus=4, batch=64 → 각 GPU batch=16")
print(f"Effect적 Batch Magnitude: {dp.n_gpus * 16} = 64")


## 28.3 All-Reduce 통신

각 GPU가 자기 그래디언트를 모든 GPU에 전파하는 통신 패턴:
- **Ring All-Reduce**: $2(N-1)$ 스텝, 각 스텝은 $|\theta|/N$ 전송
- **Tree All-Reduce**: 로그 단계

통신량: $O(|\theta|)$ per step. 7B 모델이면 매 스텝 14GB 전송.

이 통신이 병목 → 통신-연산 오버랩으로 숨김.


In [ ]:
# All-Reduce 시뮬레이션
def all_reduce_sum(grads_list):
    """각 GPU의 Gradient를 Sum산하여 모든 GPU에 분배."""
    # Sum산
    total = sum(grads_list)
    # 모든 GPU에 같은 Value 할당
    return [total.clone() for _ in grads_list]

# 4개 GPU 시뮬레이션
torch.manual_seed(0)
n_gpus = 4
local_grads = [torch.randn(5) for _ in range(n_gpus)]
print("각 GPU 로컬 그래디언트:")
for i, g in enumerate(local_grads):
    print(f"  GPU {i}: {g.numpy().round(3)}")

# All-Reduce (Mean)
reduced = all_reduce_sum(local_grads)
avg = [r / n_gpus for r in reduced]
print("\nAll-Reduce 후 (평균):")
for i, g in enumerate(avg):
    print(f"  GPU {i}: {g.numpy().round(3)}")
print("\n=> 모든 GPU가 같은 Mean Gradient를 가짐.")


## 28.4 PyTorch DDP (개념)

실제 PyTorch 코드:
```python
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP

dist.init_process_group('nccl', rank=rank, world_size=world_size)
model = model.to(rank)
model = DDP(model, device_ids=[rank])
# 이후 model(x)는 자동으로 분산 처리
```

Colab 단일 GPU에서는 실제 다중 GPU DDP를 실행할 수 없지만, gradient accumulation으로 비슷한 효과 시뮬레이션 가능.


In [ ]:
# 단일 GPU DDP 시뮬레이션
import time
from llm_math.bench import time_fn

# 큰 모델로 DDP vs 단일 배치 비교
def make_model(d=512):
    return nn.Sequential(
        nn.Linear(d, d * 2), nn.ReLU(),
        nn.Linear(d * 2, d), nn.ReLU(),
        nn.Linear(d, d)
    )

# 단일 큰 배치
torch.manual_seed(0)
model_single = make_model()
opt_single = torch.optim.AdamW(model_single.parameters(), lr=1e-3)

def step_single(x, y):
    opt_single.zero_grad()
    out = model_single(x)
    loss = F.mse_loss(out, y)
    loss.backward()
    opt_single.step()
    return loss

# DDP 시뮬레이션 (4 GPU, 각각 1/4 배치)
torch.manual_seed(0)
model_ddp = make_model()
opt_ddp = torch.optim.AdamW(model_ddp.parameters(), lr=1e-3)

def step_ddp(x, y, n_gpus=4):
    opt_ddp.zero_grad()
    x_chunks = torch.chunk(x, n_gpus)
    y_chunks = torch.chunk(y, n_gpus)
    total_loss = 0
    for xc, yc in zip(x_chunks, y_chunks):
        out = model_ddp(xc)
        loss = F.mse_loss(out, yc) / n_gpus  # 평균
        loss.backward()  # 그래디언트 누적
        total_loss += loss.item() * n_gpus
    opt_ddp.step()
    return total_loss / n_gpus

# 시간 비교 (같은 총 배치 크기)
d = 512
batch_size = 128
x = torch.randn(batch_size, d)
y = torch.randn(batch_size, d)

t_single = time_fn(step_single, x, y, device='cpu', warmup=2, repeat=5)
t_ddp = time_fn(step_ddp, x, y, device='cpu', warmup=2, repeat=5)
print(f"단일 큰 Batch (B=128): {t_single['mean_ms']:.3f} ms")
print(f"DDP 시뮬레이션 (4 GPU, 각 B=32): {t_ddp['mean_ms']:.3f} ms")
print("\n=> 단일 GPU에서는 DDP 시뮬레이션이 더 느림 (sequential 처리).")
print("   실제 다중 GPU에서는 병렬 처리로 훨씬 빠름.")


## 28.5 Large Batch Training — LARS, LAMB

큰 배치에서 학습 불안정. 층별 학습률 조정으로 해결:

**LARS** (Layer-wise Adaptive Rate Scaling):
$$\eta_\ell = \eta \cdot \frac{\|\theta_\ell\|}{\|\nabla \mathcal{L}_\ell\| + \lambda \|\theta_\ell\|}$$

**LAMB**: LARS + Adam. LLM 대규모 학습에 사용.

## 28.6 통신-연산 오버랩

역전파는 층별로 진행. 한 층의 그래디언트가 계산되면 **즉시 All-Reduce 시작**, 다음 층 역전파와 병렬.

이로써 통신 시간을 거의 숨길 수 있음.

## 28.7 핵심 정리

| 개념 | 설명 |
|---|---|
| 데이터 병렬 | 같은 모델 N개, 데이터 분할 |
| All-Reduce | 그래디언트 동기화 |
| 효과적 배치 | $N \cdot B_{\text{local}}$ |
| 통신량 | $O(\|\theta\|)$ per step |
| LARS/LAMB | 큰 배치 학습 안정화 |

## 연습문제

1. 4-GPU DDP 시뮬레이션에서 각 GPU가 다른 데이터로 학습한 후 그래디언트가 동일함을 보이라.
2. 효과적 배치 크기 32, 64, 128, 256으로 학습하고 수렴을 비교하라.
3. All-Reduce의 통신량이 $O(\|\theta\|)$인 이유를 설명하라.
4. 통신-연산 오버랩이 왜 속도를 높이는지 설명하라.
5. LARS가 왜 큰 배치에서 도움이 되는지 설명하라.

> 해설: `solutions/ch28_solutions.ipynb`
